# SmolVLA ファインチューニング: トマト山積みコンテナ掴み100ep（ACT比較用）

**目的**: ACT(act_tomato_crate_100ep)と**同一の100エピソード**でSmolVLA(450M)をLoRAファインチューニングし、
「小規模デモの単一タスクで ACT vs 基盤VLA のどちらが強いか」を実機成功率で比較する。
牡蠣v1の実測「60デモのSmolVLAは言語切替に失敗(ACT優位)」の続編: 100デモ・単一タスクならどうか。

**比較の鉄則**: 同じデータ・同じシーン・同じ評価プロトコル(各n≥10回・コンテナをゾーン分けして均等出題・成功率＋掴み所要時間)。

**事前準備**
1. `so101_tomato_crate_1cam_100ep.zip` がGoogleドライブ `MyDrive/so101/` にアップロード済みであること(ACT 100ep学習と同じzip)
2. **A100ランタイム推奨**(50,000ステップ)
3. 上から順にセルを実行

**メモ**: データセットのタスク文は `pick up a tomato from the green crate and lift it up`。
SmolVLAはこの言語指示を条件として使うため、実機推論時も同じ文字列を渡すこと(tomato_sort.pyのTASKは設定済み)。


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ZIP = "/content/drive/MyDrive/so101/so101_tomato_crate_1cam_100ep.zip"

!mkdir -p /content/dataset
!unzip -q "$DRIVE_ZIP" -d /content/dataset
!ls /content/dataset

In [ ]:
# lerobot本体 + PEFT(LoRA)ライブラリ + SmolVLAのプロセッサに必要なnum2words
# torchaoはColab既存のバージョン(0.10.0)が古すぎてPEFTの要求(0.16.0以上)を満たさないため明示的にアップグレード
!pip install -q "lerobot[dataset,training]==0.6.0" accelerate peft num2words "torchao>=0.16.0"

In [ ]:
import glob

candidates = glob.glob("/content/dataset/so101_tomato_crate_1cam_*")
assert len(candidates) == 1, f"データセットフォルダが1つに絞れません: {candidates}"
DATASET_ROOT = candidates[0]
print("DATASET_ROOT =", DATASET_ROOT)
# 100エピソード全部使用(除外なし)


## 本番ラン（A100推奨・50,000ステップ・ACT比較用）


In [ ]:
!lerobot-train \
  --dataset.repo_id=local/so101_tomato_crate_1cam \
  --dataset.root="$DATASET_ROOT" \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.load_vlm_weights=true \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --peft.r=16 \
  --output_dir=/content/drive/MyDrive/so101/train_output/smolvla_tomato_crate_100ep \
  --job_name=smolvla_tomato_crate_100ep \
  --steps=50000 \
  --batch_size=8 \
  --save_freq=5000 \
  --log_freq=50 \
  --wandb.enable=false


## 完了確認


In [ ]:
!ls -la /content/drive/MyDrive/so101/train_output/smolvla_tomato_crate_100ep/checkpoints
